# Testing-Effect Question Generation (QG) — Data Prep

No existing implementation to convert here — `src/models/` and `src/train/`
are still just `.gitkeep` placeholders, and the Obsidian project Hub only
had this as a TODO ("testing effect (QG, C) · confirm remaining pipeline
stages' team-member assignment"). This notebook builds the QG training data
from scratch, mirroring `04_rehearsal_maintenance_prep.ipynb`'s structure.

- **input** = `generate question: answer: {answer} context: {context}` — the
  reverse of normal QA (given context + question, produce an answer): here the
  model is given the context and a specific answer span, and must produce a
  question whose answer is that span. This is what lets the testing-effect
  stage generate quiz questions targeting a specific fact in a chunk. Built by
  `src/pipeline/qg.py`'s `format_qg_input`, which is the single definition of
  this format for every stage that uses it.
- **target** = the reference question.

Uses `squad` (v1.1) — SQuAD ships (context, question, answer) triples
directly, so no oracle-extraction step is needed the way `cnn_dailymail`'s
abstractive-only summaries required for the A/B rehearsal data (notebook
`04`).

**Context length vs. chunk scale.** SQuAD contexts are Wikipedia paragraphs
running 143 / 239 / 312 words at the 50th / 90th / 99th percentile. That sat
entirely under `configs/chunking.yaml`'s old `max_words = 350`, but the config
was corrected to **200** on 2026-07-30 (350 overflowed RoBERTa's 512-token
ceiling on every genre — see the config's own comment table). So the upper
half of SQuAD's contexts are now *longer* than the chunks this model will
actually be asked about at inference.

No truncation step is added for it, deliberately: training on somewhat longer
contexts than inference will show is the safe direction of the mismatch — the
model sees the shorter form as an easier case, not an unfamiliar one. Worth
re-checking if question quality turns out to depend on context length.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

from src.pipeline.qg import format_qg_input

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load `squad`

`squad` (v1.1) has official `train`/`validation` splits with `context`,
`question`, `answers` (`{"text": [...], "answer_start": [...]}`) fields.
Each training row has exactly one answer; we just take `answers.text[0]`.

SQuAD has no public `test` split — the real test answers are held for the
official leaderboard. So `validation` is split in half: the first half drives
checkpoint selection (`val`), the second is touched once at the end of `09`
(`test`). Train and validation titles are **fully disjoint** (442 vs 48, zero
overlap), so val/test already measure transfer to unseen articles.

> [!danger] 2026-08-01 — the previous slicing drew from almost no articles
> **SQuAD rows are ordered by title**, and this notebook used `train[:3000]` /
> `validation[:600]`. Measured:
>
> | split | rows | distinct titles (before) |
> |---|---|---|
> | train | 3,000 | **9** |
> | val + test | 600 | **1** |
>
> So the QG model was trained on 3,000 questions drawn from nine Wikipedia
> articles (Beyoncé, Chopin, Antibiotics, iPod, Montana, …), and its reported
> val/test ROUGE-L was measured on a **single** article. That is a much
> narrower distribution than "SQuAD is Wikipedia" implies, and the likeliest
> explanation for the out-of-domain failures seen in `09` §6 — which were
> tentatively blamed on genre shift (wiki → news) when the real training
> distribution was nine specific pages.
>
> **Fix**: shuffle before slicing, and draw training rows only from the
> `qg_titles` pool that `build_wiki_train_corpora.py` reserved. All previously
> reported QG numbers are void.

### Why the title pool matters beyond diversity

`data/processed/wiki_train/title_partition.json` splits SQuAD's 442 titles
three ways, with no overlap:

| partition | titles | used for |
|---|---|---|
| `eval_title` | 1 | the wiki eval corpus (`New_York_City`) |
| `corpus_titles` | 217 | the 20 wiki train/reward corpora |
| `qg_titles` | 224 | **QG training, here** |

Without this split, retraining QG on more SQuAD would eventually have it train
on the very wiki documents it is later asked to generate probes about — and
those probes feed B's stage-2 curation, so the contamination would propagate
into B's training targets rather than staying a QG problem.

In [2]:
import json

MAX_TRAIN_EXAMPLES = 3000
MAX_VAL_EXAMPLES = 300
MAX_TEST_EXAMPLES = 300
SEED = 20260801

# Titles reserved for QG by build_wiki_train_corpora.py — disjoint from the
# wiki eval corpus and from all 20 wiki train corpora, so the QG model never
# trains on a document it will later be asked to generate probes for.
partition = json.loads(
    Path("data/processed/wiki_train/title_partition.json").read_text(encoding="utf-8")
)
qg_titles = set(partition["qg_titles"])
print(f"QG title pool: {len(qg_titles)} titles "
      f"(eval 1 / wiki corpora {len(partition['corpus_titles'])} excluded)")

train_full = load_dataset("squad", split="train")
train_pool = train_full.filter(lambda row: row["title"] in qg_titles)

# Shuffle before slicing. SQuAD is ordered by title, so train[:3000] draws from
# only ~9 of the 442 titles — a far narrower training distribution than "SQuAD
# is Wikipedia" suggests, and the likeliest cause of the QG model's poor
# out-of-domain behaviour in `09`.
train_raw = train_pool.shuffle(seed=SEED).select(range(MAX_TRAIN_EXAMPLES))

# Same problem on the eval side: validation[:600] covers exactly ONE title, so
# the previously reported val/test scores were measured on a single article.
val_test_raw = (
    load_dataset("squad", split="validation")
    .shuffle(seed=SEED)
    .select(range(MAX_VAL_EXAMPLES + MAX_TEST_EXAMPLES))
)
val_raw = val_test_raw.select(range(MAX_VAL_EXAMPLES))
test_raw = val_test_raw.select(range(MAX_VAL_EXAMPLES, len(val_test_raw)))

print(f"\ntrain: {len(train_raw)} (pool {len(train_pool):,}), "
      f"validation: {len(val_raw)}, test: {len(test_raw)}")
for name, split in [("train", train_raw), ("val", val_raw), ("test", test_raw)]:
    print(f"  {name:5} covers {len(set(split['title'])):3} distinct titles")

# SQuAD's train and validation titles are fully disjoint (442 vs 48, no
# overlap), so val/test already measure transfer to unseen articles.
assert not (set(train_raw["title"]) & set(val_raw["title"])), "train/val title leak"
assert not (set(train_raw["title"]) & set(test_raw["title"])), "train/test title leak"
assert partition["eval_title"] not in set(train_raw["title"]), "wiki eval title leaked into QG training"

sample = train_raw[0]
print("\nsample title:", sample["title"])
print("question:", sample["question"])
print("answers:", sample["answers"])

QG title pool: 224 titles (eval 1 / wiki corpora 217 excluded)


Filter: 100%|██████████| 87599/87599 [00:00<00:00, 142310.62 examples/s]



train: 3000 (pool 45,399), validation: 300, test: 300
  train covers 223 distinct titles
  val   covers  48 distinct titles
  test  covers  48 distinct titles

sample title: Turner_Classic_Movies
question: Who began to host Essentials Jr. in 2011?
answers: {'text': ['Bill Hader'], 'answer_start': [250]}


## 2. Build (input, target) pairs — reversed QA

`input_text = "generate question: answer: {answer} context: {context}"`,
`target_text = question`.

The template lives in **`src/pipeline/qg.py` as `format_qg_input`**, not as an
f-string here, because the same string has to be produced at generation time
in `09` and later by the stage-2 curation loop that uses QG probes. A template
copied into three places drifts, and this one already did — see the warning
below.

The `generate question:` prefix: T5 was pretrained with task prefixes
(`summarize:`, `translate English to German:`), and the pilot run without one
produced declarative sentences rather than interrogatives — it echoed the
context instead of asking about it. Undertraining is the more likely cause
(200 examples, 1 epoch), but the prefix is free insurance and standard T5
practice for single-task fine-tuning. Rows with no answer text (shouldn't
happen in SQuAD v1.1's train/validation splits, but checked defensively) are
skipped.

In [3]:
def build_pair(example: dict) -> dict | None:
    answer_texts = example["answers"]["text"]
    if not answer_texts:
        return None
    return {
        "id": example["id"],
        # format_qg_input, not an inline f-string: `09` and the stage-2
        # curation loop have to produce the byte-identical form at generation
        # time, and a template copied into three places drifts silently.
        "input_text": format_qg_input(answer_texts[0], example["context"]),
        "target_text": example["question"],
    }


train_pairs = [p for p in (build_pair(ex) for ex in train_raw) if p is not None]
val_pairs = [p for p in (build_pair(ex) for ex in val_raw) if p is not None]
test_pairs = [p for p in (build_pair(ex) for ex in test_raw) if p is not None]
print(f"train pairs: {len(train_pairs)} / {len(train_raw)}")
print(f"val pairs: {len(val_pairs)} / {len(val_raw)}")
print(f"test pairs: {len(test_pairs)} / {len(test_raw)}")

assert train_pairs[0]["input_text"].startswith("generate question:"), "task prefix missing"
print("\nsample input:", train_pairs[0]["input_text"][:110], "...")
pd.DataFrame(train_pairs)[["input_text", "target_text"]].head(3)

train pairs: 3000 / 3000
val pairs: 300 / 300
test pairs: 300 / 300

sample input: generate question: answer: Bill Hader context: "Funday Night at the Movies" was replaced in 2008 by "Essential ...


,input_text,target_text
0,generate question: answer: Bill Hader context:...,Who began to host Essentials Jr. in 2011?
1,generate question: answer: hereditary juridica...,What ultimately determined nobility?
2,generate question: answer: Abu Salim context: ...,At what prison did extrajudicial executions oc...


## 3. Tokenize

`INPUT_MAX_LENGTH`/`TARGET_MAX_LENGTH` are set from measured token-length
percentiles with the `t5-small` tokenizer: input (`answer: ... context:
...`) runs ~213/359/513 tokens at the 50th/90th/99th percentile, and
questions run ~16/22/26. 512 covers the bulk of inputs (a handful of the
longest contexts get truncated at the tail — acceptable for a pilot); 32
comfortably covers question length.

In [4]:
INPUT_MAX_LENGTH = 512
TARGET_MAX_LENGTH = 32

tokenizer = AutoTokenizer.from_pretrained("t5-small")


def tokenize_pairs(pairs: list[dict]) -> datasets.Dataset:
    inputs = tokenizer(
        [p["input_text"] for p in pairs],
        max_length=INPUT_MAX_LENGTH,
        truncation=True,
    )
    targets = tokenizer(
        [p["target_text"] for p in pairs],
        max_length=TARGET_MAX_LENGTH,
        truncation=True,
    )
    return datasets.Dataset.from_dict(
        {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": targets["input_ids"],
        }
    )


train_dataset = tokenize_pairs(train_pairs)
val_dataset = tokenize_pairs(val_pairs)
test_dataset = tokenize_pairs(test_pairs)
print(train_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})


## 4. Save

`notebooks/09_qg_testing_effect_train.ipynb` loads these directly —
train+val during training, test only once at the end for final reporting.

In [5]:
import shutil

OUT_DIR = Path("data/processed/qg_testing_effect")

# Clear first. An earlier version of this notebook wrote a prefix-less dataset
# here, and a partial overwrite would leave splits in mixed formats — which
# fails silently, as a quality problem rather than an error.
if OUT_DIR.exists():
    print(f"Removing existing {OUT_DIR} so no stale split survives")
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset.save_to_disk(str(OUT_DIR / "train"))
val_dataset.save_to_disk(str(OUT_DIR / "val"))
test_dataset.save_to_disk(str(OUT_DIR / "test"))

pd.DataFrame(train_pairs).to_csv(OUT_DIR / "train_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(val_pairs).to_csv(OUT_DIR / "val_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(test_pairs).to_csv(OUT_DIR / "test_pairs_raw.csv", index=False, encoding="utf-8-sig")

print(f"Saved to: {OUT_DIR}")
for name, ds in [("train", train_dataset), ("val", val_dataset), ("test", test_dataset)]:
    decoded = tokenizer.decode(ds[0]["input_ids"], skip_special_tokens=True)
    assert decoded.startswith("generate question:"), f"{name} written without the task prefix"
    print(f"  {name}: {len(ds)} rows | prefix OK")

Removing existing data/processed/qg_testing_effect so no stale split survives


Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 133039.88 examples/s]

Saved to: data/processed/qg_testing_effect
  train: 3000 rows | prefix OK
  val: 300 rows | prefix OK
  test: 300 rows | prefix OK


## Summary

- Source: `squad` (v1.1), drawn from the **`qg_titles` pool (224 titles)** in
  `data/processed/wiki_train/title_partition.json` — disjoint from the wiki
  eval corpus and from all 20 wiki train corpora.
- **Shuffled before slicing** (seed 20260801). SQuAD is title-ordered, so the
  previous `train[:3000]` / `validation[:600]` covered only 9 and 1 titles
  respectively. Every QG number reported before 2026-08-01 is void.
- `validation` split in half into `val` (checkpoint selection) and `test`
  (touched once, at the end of `09`) — SQuAD has no public test split. Train
  and validation titles are fully disjoint by construction of the dataset.
- No oracle-extraction step needed (unlike `04`'s rehearsal data) — SQuAD
  gives (context, question, answer) directly; we reverse the usual QA
  direction via `src/pipeline/qg.py`'s `format_qg_input`.
- Context length: SQuAD paragraphs run 143/239/312 words at the 50/90/99th
  percentile, against `configs/chunking.yaml`'s `max_words = 200`. The upper
  half is longer than the chunks this model will see at inference — left
  untruncated deliberately, since training on longer contexts than inference
  is the safe direction of the mismatch.
- Next: re-run `09_qg_testing_effect_train.ipynb`. Its §1 asserts the task
  prefix is present, and its §5 probe-quality cell measures answer
  conditioning and answer leakage — the numbers to compare against the
  previous run (test ROUGE-L 0.3461, leakage 7.3%, cross-answer ROUGE-L
  0.3268) to see whether the narrow title pool was the real cause.